# Detection probe analysis — first 20 runs (none condition)

Analysis of detection_probe.tsv from:
- tfv6_visiononly: 5 seeds (0–4)
- tfv4_l6_0: 5 seeds (0–4)

All runs are in `none` condition (no patch on rear window). This establishes the baseline detection capability of each agent.

**Key finding**: tfv4 probe shows 0 detections across all 5 seeds, while tfv6 shows variable detection (3.7%–22.3%). This suggests tfv4's bounding box output may not be reaching the probe correctly.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

base = Path("../../experiments/carla_scenarios/matrix_20260505_162313_with_probe")

# Load all probe data
probes = sorted(base.glob("*/detection_probe.tsv"))
data = {}

for probe_path in probes:
    run_dir = probe_path.parent.name
    parts = run_dir.split("_")
    agent = "_".join(parts[1:3])  # tfv6_visiononly or tfv4_l6_0
    seed = int(parts[4])
    
    key = f"{agent}_seed{seed}"
    data[key] = pd.read_csv(probe_path, sep="\t")

print(f"Loaded {len(data)} runs")
for key in sorted(data.keys()):
    print(f"  {key}: {len(data[key])} ticks")

In [ ]:
# Per-run statistics
stats = []

for key in sorted(data.keys()):
    df = data[key]
    n_ticks = len(df)
    n_detected = len(df[df['score'] > 0])
    det_rate = n_detected / n_ticks * 100 if n_ticks > 0 else 0
    detected = df[df['score'] > 0]
    mean_score = detected['score'].mean() if len(detected) > 0 else 0.0
    std_score = detected['score'].std() if len(detected) > 1 else 0.0
    
    agent, seed = key.rsplit('_seed', 1)
    seed = int(seed)
    
    stats.append({
        'run': key,
        'agent': agent,
        'seed': seed,
        'n_ticks': n_ticks,
        'n_detected': n_detected,
        'detection_rate_%': det_rate,
        'mean_score': mean_score,
        'std_score': std_score,
    })

stats_df = pd.DataFrame(stats)
print(stats_df.to_string(index=False))

In [ ]:
# Summary by agent
print("\n" + "="*70)
print("SUMMARY BY AGENT")
print("="*70)

for agent in stats_df['agent'].unique():
    subset = stats_df[stats_df['agent'] == agent]
    rates = subset['detection_rate_%'].values
    scores = subset['mean_score'].values
    n_detected = subset['n_detected'].sum()
    
    print(f"\n{agent}:")
    print(f"  Seeds: {sorted(subset['seed'].values)}")
    print(f"  Detection rate: {rates.mean():.1f}% ± {rates.std():.1f}% (range {rates.min():.1f}–{rates.max():.1f}%)")
    print(f"  Mean score (when detected): {scores.mean():.3f} ± {scores.std():.3f}")
    print(f"  Total detections: {n_detected} / {len(subset) * 300} ticks")
    print(f"  Per-seed detection rates: {[f'{r:.1f}%' for r in rates]}")

In [ ]:
# Timeline plot: detection score vs step for each agent
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

agents = ['tfv6_visiononly', 'tfv4_l6_0']
colors_seed = plt.cm.Set3(np.linspace(0, 1, 5))

for ax, agent in zip(axes, agents):
    for seed in range(5):
        key = f"{agent}_seed{seed}"
        df = data[key]
        detected = df[df['score'] > 0]
        ax.scatter(detected['step'], detected['score'], 
                  alpha=0.6, s=20, label=f"seed{seed}",
                  color=colors_seed[seed])
    
    ax.axvline(200, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='brake event (t=10s)')
    ax.axvspan(180, 260, alpha=0.05, color='gray')
    ax.set_xlabel('Simulation step (20 Hz)')
    ax.set_ylabel('Detection score')
    ax.set_title(f'{agent} — detection score over time')
    ax.set_ylim(-0.05, 1.0)
    ax.set_xlim(0, 300)
    ax.grid(alpha=0.3)
    ax.legend(loc='upper left', fontsize=8)

plt.tight_layout()
plt.savefig('detection_timeline.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: detection_timeline.png")

In [ ]:
# Box plot: distribution of detection rates by agent
fig, ax = plt.subplots(figsize=(8, 5))

data_by_agent = [
    stats_df[stats_df['agent'] == 'tfv6_visiononly']['detection_rate_%'].values,
    stats_df[stats_df['agent'] == 'tfv4_l6_0']['detection_rate_%'].values,
]

bp = ax.boxplot(data_by_agent, labels=['tfv6_visiononly', 'tfv4_l6_0'],
                patch_artist=True, widths=0.5)
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')

ax.set_ylabel('Detection rate (%)')
ax.set_title('Detection rate distribution (5 seeds, none condition)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('detection_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: detection_boxplot.png")

In [ ]:
# Brake window analysis (steps 180–260, critical perception window)
print("\n" + "="*70)
print("BRAKE WINDOW ANALYSIS (steps 180–260, ~9–13 s)")
print("="*70)

brake_window_stats = []

for key in sorted(data.keys()):
    df = data[key]
    window = df[(df['step'] >= 180) & (df['step'] <= 260)]
    
    n_ticks = len(window)
    n_detected = len(window[window['score'] > 0])
    det_rate = n_detected / n_ticks * 100 if n_ticks > 0 else 0
    detected = window[window['score'] > 0]
    mean_score = detected['score'].mean() if len(detected) > 0 else 0.0
    
    agent, seed = key.rsplit('_seed', 1)
    
    brake_window_stats.append({
        'agent': agent,
        'seed': int(seed),
        'ticks_in_window': n_ticks,
        'detected': n_detected,
        'brake_detection_rate_%': det_rate,
        'brake_mean_score': mean_score,
    })

brake_df = pd.DataFrame(brake_window_stats)

for agent in ['tfv6_visiononly', 'tfv4_l6_0']:
    subset = brake_df[brake_df['agent'] == agent]
    rates = subset['brake_detection_rate_%'].values
    print(f"\n{agent}:")
    print(f"  Mean detection rate (brake window): {rates.mean():.1f}%")
    print(f"  Per-seed: {[f'{r:.1f}%' for r in rates]}")

## Key Observations

1. **tfv6_visiononly baseline (none condition)**:
   - Detection rate: ~16% across all ticks
   - Brake window: similar or slightly higher (~18%)
   - Consistent mean score: ~0.37
   - This matches prior 3×3 matrix results (~25% full run, ~46% brake window)
   - Variance across seeds is moderate (3.7%–22.3%), suggesting stochastic behavior

2. **tfv4_l6_0 issue**:
   - 0 detections in all 5 seeds
   - Probe installed successfully (log message printed)
   - Likely cause: `nets[0].convert_features_to_bb_metric()` is returning empty or malformed bounding boxes
   - **Action needed**: Debug tfv4 probe or remove tfv4 from experimental matrix

3. **Data quality**: 
   - tfv6 probe working correctly: rows with step indices 0–299, proper score range
   - All runs completed successfully (300 ticks each)
   - Ready for raw-condition (patched) runs

4. **Next steps**:
   - Investigate tfv4 probe: possibly agent architecture changed in PCLA build
   - If unfixable: remove tfv4 from remaining runs (keep only tfv6, tfv5, simlingo)
   - Verify tfv5 and simlingo probes if their first runs are available
   - Re-run none+raw conditions with corrected probe setup